# (노트) 순환신경망

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [빅데이터분석]

In [5]:
from fastai.text.all import *
import torch 
import numpy as np

In [6]:
text= 'h e l l o '*100
text

'h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o h e l l o

In [7]:
tokens= text.split(' ')[:-1]
tokens[:10]

['h', 'e', 'l', 'l', 'o', 'h', 'e', 'l', 'l', 'o']

`-` 바로직전문자로 다음문자를 맞춰보자. 
- hello 니까, h $\to$ e, e $\to$ l, l $\to$ l/o (?), l $\to$ o, o $\to$ h ... 
- l 다음에 올 문자가 조금 애매하다. 

`-` 마치 아래의 표에서 $X \to y$인 맵핑을 알아차려 $X$를 보고 $y$를 예측하듯이 

|X|y|
|:-:|:-:|
|1|2|
|2|4|
|3|6|
|1|2|
|2|4|
|3|6|
|...|...|

우리는 이제 아래를 예측하는 규칙을 알아차리는게 목표이다. 

|X|y|
|:-:|:-:|
|h|e|
|e|l|
|l|l/o|
|o|h|
|h|e|
|...|...|

`-` X,y를 설정하자.  

In [8]:
X= tokens[:(len(tokens)-1)]
y= tokens[1:len(tokens)]

In [9]:
X[0],y[0]

('h', 'e')

In [10]:
X[1],y[1]

('e', 'l')

`-` 다 좋은데 학습가능한 형태로 만들어야 하므로 문자를 숫자로 바꾸자. 

In [11]:
dic = {'h': 0, 'e': 1, 'l': 2, 'o': 3}
dic

{'h': 0, 'e': 1, 'l': 2, 'o': 3}

In [12]:
dic['h'],dic['e'],dic['l'],dic['o'] 

(0, 1, 2, 3)

In [13]:
nums = [dic[i] for i in tokens]

In [14]:
tokens[:10],nums[:10]

(['h', 'e', 'l', 'l', 'o', 'h', 'e', 'l', 'l', 'o'],
 [0, 1, 2, 2, 3, 0, 1, 2, 2, 3])

`-` 아래와 같이 문자와 숫자를 mapping 하였다.

|문자(tokens)|숫자(nums)|
|:-:|:-:|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|'h'|0|
|'e'|1|
|'l'|2|
|'l'|2|
|'o'|3|
|...|...|

`-` 이제 숫자화된 자료를 이용하여 다시 X,y를 선언하자. 

In [15]:
X = torch.tensor(nums[:499])
y = torch.tensor(nums[1:])

In [16]:
X[0],y[0]

(tensor(0), tensor(1))

In [17]:
X[1],y[1]

(tensor(1), tensor(2))

`-` 간단한 네트워크를 통하여 학습을 시도하자. 

In [18]:
e1=torch.nn.Embedding(num_embeddings=4,embedding_dim=20) 
l1=torch.nn.Linear(in_features=20,out_features=20)
a1=torch.nn.ReLU()
l2=torch.nn.Linear(in_features=20,out_features=4)
a2=torch.nn.Softmax()

In [19]:
X.shape,e1(X).shape

(torch.Size([499]), torch.Size([499, 20]))

In [20]:
e1(X).shape,a1(l1(e1(X))).shape

(torch.Size([499, 20]), torch.Size([499, 20]))

In [21]:
a1(l1(e1(X))).shape,l2(a1(l1(e1(X)))).shape

(torch.Size([499, 20]), torch.Size([499, 4]))

In [22]:
l2(a1(l1(e1(X))))[0]

tensor([-0.1281, -0.6513, -0.0344,  0.1512], grad_fn=<SelectBackward0>)

In [23]:
a2(l2(a1(l1(e1(X)))))[0]

<ipython-input-23-17d00ccf79de>:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  a2(l2(a1(l1(e1(X)))))[0]


tensor([0.2492, 0.1477, 0.2737, 0.3295], grad_fn=<SelectBackward0>)

- $X$의 차원이 명시되지 않아서 대충 내가 알아서 계산했다는 워닝

`-` 경고가 찝찝하여 손계산해봄 $\to$ 알아서 잘 계산했음 

In [24]:
np.exp(0.1195)/(np.exp(0.1195)+np.exp(-0.3600)+np.exp(-0.1282)+np.exp(-0.5230))

0.34180289149024373

`-` 순전파차원변화 요약 

```
torch.Size([499]) ## X with levels=4 
torch.Size([499, 20]) ## 임베딩 이후
torch.Size([499, 20]) ## 선형변환1 이후
torch.Size([499, 20]) ## 렐루 이후
torch.Size([499, 4]) ## 선형변환2 이후 
torch.Size([499, 4]) ## 소프트맥스 이후 = yhat 
```

In [25]:
net = torch.nn.Sequential(
    torch.nn.Embedding(num_embeddings=4,embedding_dim=20),
    torch.nn.Linear(in_features=20,out_features=20),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=20,out_features=4))
    #torch.nn.Softmax()

In [26]:
net(X)

tensor([[ 0.1517, -0.1398,  0.1700,  0.1867],
        [ 0.0241, -0.3839,  0.2325,  0.5485],
        [ 0.1313, -0.0685,  0.1634, -0.1837],
        ...,
        [ 0.0241, -0.3839,  0.2325,  0.5485],
        [ 0.1313, -0.0685,  0.1634, -0.1837],
        [ 0.1313, -0.0685,  0.1634, -0.1837]], grad_fn=<AddmmBackward0>)

- 순전파 

`-` 손실함수, 옵티마이저 

In [27]:
loss_fn = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(net.parameters())

In [28]:
for i in range(1000):
    ## 1 
    yhat = net(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward()
    ## 4 
    optimizer.step()
    optimizer.zero_grad()

In [29]:
dic

{'h': 0, 'e': 1, 'l': 2, 'o': 3}

In [30]:
X[:7] 

tensor([0, 1, 2, 2, 3, 0, 1])

In [31]:
net(X)[:7]

tensor([[-1.9828,  5.7060, -1.8155, -2.7688],
        [-4.0879, -2.8654,  5.2053, -2.8722],
        [-3.4130, -3.3800,  4.4111,  4.4107],
        [-3.4130, -3.3800,  4.4111,  4.4107],
        [ 4.2052, -3.5282, -3.8988, -3.2520],
        [-1.9828,  5.7060, -1.8155, -2.7688],
        [-4.0879, -2.8654,  5.2053, -2.8722]], grad_fn=<SliceBackward0>)

In [32]:
y[:7]

tensor([1, 2, 2, 3, 0, 1, 2])

- 학습이 잘 되었다. 

### net의 개선 

`-` 단어수가 바뀔때마다 아래를 새로 정의해야 하나?
```python
net = torch.nn.Sequential(
    torch.nn.Embedding(num_embeddings=4,embedding_dim=20),
    torch.nn.Linear(in_features=20,out_features=20),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=20,out_features=4))
    #torch.nn.Softmax()
```

`-` 네트워크를 찍어내는 뭔가가 있음 좋지 않을까? 제가 만들어볼게요!

In [33]:
class BDA(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        u= self.linear1(self.embedding(X))
        v= self.relu(u)
        return self.linear2(v)

- 그냥 이렇게 간단해보이는 코드로 가능하다고? $\to$ 필요한 다른기능들은 Module에서 상속받았으므로 가능하다!

In [34]:
net2 = BDA(4) # 4는 dic의 size

In [35]:
net

Sequential(
  (0): Embedding(4, 20)
  (1): Linear(in_features=20, out_features=20, bias=True)
  (2): ReLU()
  (3): Linear(in_features=20, out_features=4, bias=True)
)

In [36]:
net2

BDA(
  (embedding): Embedding(4, 20)
  (linear1): Linear(in_features=20, out_features=20, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=20, out_features=4, bias=True)
)

- 오.. 

In [37]:
loss_fn = torch.nn.CrossEntropyLoss() 
optimizer2 = torch.optim.Adam(net2.parameters())

In [38]:
for i in range(1000):
    ## 1 
    yhat = net2(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward()
    ## 4 
    optimizer2.step()
    optimizer2.zero_grad()

In [39]:
net2(X)

tensor([[-3.4576,  4.5350, -4.1240, -4.0303],
        [-3.6154, -5.0942,  4.9586, -1.8321],
        [-4.0719, -3.9697,  3.2151,  3.2140],
        ...,
        [-3.6154, -5.0942,  4.9586, -1.8321],
        [-4.0719, -3.9697,  3.2151,  3.2140],
        [-4.0719, -3.9697,  3.2151,  3.2140]], grad_fn=<AddmmBackward0>)

In [40]:
net(X)

tensor([[-1.9828,  5.7060, -1.8155, -2.7688],
        [-4.0879, -2.8654,  5.2053, -2.8722],
        [-3.4130, -3.3800,  4.4111,  4.4107],
        ...,
        [-4.0879, -2.8654,  5.2053, -2.8722],
        [-3.4130, -3.3800,  4.4111,  4.4107],
        [-3.4130, -3.3800,  4.4111,  4.4107]], grad_fn=<AddmmBackward0>)

`-` net2도 잘 학습 되었다.

### AR(2) 

In [41]:
X = torch.tensor([nums[:498],nums[1:499]]).T
y = torch.tensor(nums[2:])

In [42]:
X[0],y[0] # he -> l 

(tensor([0, 1]), tensor(2))

In [43]:
X[1],y[1] # el -> l 

(tensor([1, 2]), tensor(2))

In [44]:
X[2],y[2] # ll -> o

(tensor([2, 2]), tensor(3))

`-` 아키텍처를 대충 스케치해보자. 

In [45]:
_e1 = torch.nn.Embedding(num_embeddings=4,embedding_dim=20)

In [46]:
X.shape, _e1(X).shape

(torch.Size([498, 2]), torch.Size([498, 2, 20]))

`-` 이전의 아키텍처는 아래와 같았음. 

```
torch.Size([499]) ## X with levels=4 
torch.Size([499, 20]) ## 임베딩 이후
torch.Size([499, 20]) ## 선형변환1 이후
torch.Size([499, 20]) ## 렐루 이후
torch.Size([499, 4]) ## 선형변환2 이후 
torch.Size([499, 4]) ## 소프트맥스 이후 = yhat 
```

`-` 벌써부터 꼬이는데?? 마지막 차원은 어쩌지? 합치나? 
- 합치는게 나쁜건 아닐지 몰라도 다른 대안이 있음 $\to$ 순환신경망의 발견 

`-` 순환신경망의 설계 

In [47]:
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        x1=X[:,0] # X의 첫번째 칼럼, y보다 2시점이전
        x2=X[:,1] # X의 두번째 칼럼, y보다 1시점이전 
        v=self.relu(self.linear1(self.embedding(x1))) # v: (498,20) 
        v2=self.relu(self.linear1(v+self.embedding(x2))) # 
        return self.linear2(v2)

`-` 결국 최종출력인 v2는 v와 x2이 담긴 함수이다. 그런데 v는 x1이 담긴 함수이다. 따라서 v2는 x2가 담겨있는 동시에 x1도 약하게 담겨있다 볼 수 있음. 

In [48]:
net3=BDA2(4)
net3

BDA2(
  (embedding): Embedding(4, 20)
  (linear1): Linear(in_features=20, out_features=20, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=20, out_features=4, bias=True)
)

In [49]:
net2

BDA(
  (embedding): Embedding(4, 20)
  (linear1): Linear(in_features=20, out_features=20, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=20, out_features=4, bias=True)
)

- 구조의 차이는 없지만 순전파 계산방식이 다르다! (그렇다면 역전파 계산방식도 다르겠지?..) 

`-` 다시 학습해보자. 

In [50]:
loss_fn = torch.nn.CrossEntropyLoss() # 사실손실함수는 계속 재활용해도 괜찮은데.. 어차피 같은거임 
optimizer3 = torch.optim.Adam(net3.parameters())

In [51]:
for i in range(1000):
    ## 1 
    yhat = net3(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward() 
    ## 4 
    optimizer3.step()
    optimizer3.zero_grad()

In [52]:
X[:5]

tensor([[0, 1],
        [1, 2],
        [2, 2],
        [2, 3],
        [3, 0]])

In [53]:
net3(X)[:5]

tensor([[-2.5685, -2.9446,  5.9947, -5.8544],
        [-3.7520, -3.1265,  5.7235, -2.2666],
        [-0.4511, -5.3339, -1.0451,  7.6977],
        [ 6.2052, -4.2444, -6.0145, -1.6563],
        [-2.6199,  6.3203, -2.4867, -2.3902]], grad_fn=<SliceBackward0>)

- he $\to$ l 
- el $\to$ l 
- ll $\to$ o 
- lo $\to$ h
- oh $\to$ e 

`-` 학습이 잘되었다. 

`-` 네트워크를 만드는 클래스를 조금 정리하자. 

```python 
### 원래 이런클래스였음 
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        x1=X[:,0] # X의 첫번째 칼럼, y보다 2시점이전
        x2=X[:,1] # X의 두번째 칼럼, y보다 1시점이전 
        v=self.relu(self.linear1(self.embedding(x1))) # v: (498,20) 
        v2=self.relu(self.linear1(v+self.embedding(x2))) # 
        return self.linear2(v2)

```

```python 
### 정리해보자 
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        v=0 
        for i in range(2):        
        v=self.relu(self.linear1(v+self.embedding(X[:,i]))) # 
        return self.linear2(v)

```

In [54]:
class BDA2(Module): 
    def __init__(self, num_embeddings): 
        self.embedding = nn.Embedding(num_embeddings,20)
        self.linear1 = nn.Linear(20,20)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(20,num_embeddings) 
    def forward(self,X):
        v=0 
        for i in range(2):        
            v=self.relu(self.linear1(v+self.embedding(X[:,i]))) # 
        return self.linear2(v)

In [56]:
net3=BDA2(4)
loss_fn = torch.nn.CrossEntropyLoss() # 사실손실함수는 계속 재활용해도 괜찮은데.. 어차피 같은거임 
optimizer3 = torch.optim.Adam(net3.parameters())

In [57]:
for i in range(1000):
    ## 1 
    yhat = net3(X) 
    ## 2 
    loss = loss_fn(yhat,y) 
    ## 3 
    loss.backward() 
    ## 4 
    optimizer3.step()
    optimizer3.zero_grad()

In [58]:
X[:5]

tensor([[0, 1],
        [1, 2],
        [2, 2],
        [2, 3],
        [3, 0]])

In [59]:
net3(X)[:5]

tensor([[-3.9664, -4.8803,  4.4494, -5.6113],
        [-2.8232, -3.7962,  7.1852, -0.9583],
        [-1.9582, -4.7769, -1.7349,  6.3795],
        [ 6.2435, -2.6863, -3.4886, -1.4581],
        [-2.7031,  5.7897, -3.1194, -2.9611]], grad_fn=<SliceBackward0>)

- 정리가 잘되어서 동일한 결과가 나옴 

`-` "2개의문자 $\to$ 다음 1개 문자"와 같은 예측이 아니고 "여러개의 문자 $\to$ 다음 여러개의 문자"를 예측하고 싶다. 